In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DATA_RAW_DIR = "../../data/raw/"

In [ ]:
fci = pd.read_csv(DATA_RAW_DIR + "FCI.csv", sep=";", encoding="utf-8-sig")
coloris = pd.read_csv(DATA_RAW_DIR + "COLORIS.csv", sep=";", encoding="utf-8-sig")
aero = pd.read_csv(DATA_RAW_DIR + "AERO.csv", sep=";", encoding="utf-8-sig")
vio = pd.read_csv(DATA_RAW_DIR + "VIO.csv", sep=";", encoding="utf-8-sig")

In [ ]:
display(fci.info())
display(fci.head())

In [ ]:
display(coloris.info())
display(coloris.head())

In [ ]:
display(aero.info())
display(aero.head())

In [ ]:
display(vio.info())
display(vio.head())

## Shared helpers

Used by every per-application section below: French accented text normalization, title
truncation, the `actions_realisees` → `TransferDestination` matcher, and the `Offre composée`
offer/version parser.

In [ ]:
import re
import sys
import unicodedata

sys.path.insert(0, "../..")
from app.modules.ticket_management.domain.enums.transfer_destination import TransferDestination

DEST_NAME_TO_VALUE = {member.name: member.value for member in TransferDestination}


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.lower()


TITLE_MAX_LEN = 80


def truncate_title(description: str) -> str:
    description = description.strip()
    if len(description) <= TITLE_MAX_LEN:
        return description
    return description[: TITLE_MAX_LEN - 1].rstrip() + "…"


_DUREE_RE = re.compile(r"(Résolu|Transféré) en (?:(< 1)|(\d+)) jours?")


def parse_duree_resolution(value: str) -> tuple[str, int]:
    """Parses the 'Durée de résolution' display column into (action_word, day_count)."""
    match = _DUREE_RE.search(value)
    if match is None:
        raise ValueError(f"Unrecognized 'Durée de résolution' format: {value!r}")
    action, lt_one, days = match.groups()
    return action, 0 if lt_one else int(days)


# Applications with a support/paramétrage team split, matched via "paramétrage <app>" /
# "support <app>" (in either word order). "Catalogue" is this org's own name for the
# paramétrage/configuration team, so "catalogue <app>" / "<app> catalogue" is matched the same
# way as "paramétrage <app>".
TEAM_QUALIFIED_APPS = {
    "COLORIS": ("CONFIG_COLORIS", "SUPPORT_COLORIS"),
    "FCI": ("CONFIG_FCI", "SUPPORT_FCI"),
}

# Destinations matched as a bare keyword, using each enum member's own value. BANCO is handled
# separately below since it also shows up as an incidental root-cause mention.
SIMPLE_KEYWORD_DESTINATIONS = [
    "AERO", "VIO", "EEP", "CLIP", "ULYSSE", "ACACIA", "SANTAFE", "PROXIMA", "HABILITATION",
]


def _detect_destination(norm: str, default_catalogue_app: str) -> tuple[str | None, str | None]:
    for app, (config_name, support_name) in TEAM_QUALIFIED_APPS.items():
        app_norm = normalize_text(app)
        if re.search(rf"parametrage.{{0,20}}{app_norm}|{app_norm}.{{0,20}}parametrage", norm):
            return config_name, None
        if re.search(rf"catalogue.{{0,20}}{app_norm}|{app_norm}.{{0,20}}catalogue", norm):
            return config_name, None
        if re.search(rf"support.{{0,20}}{app_norm}", norm):
            return support_name, None
    for name in SIMPLE_KEYWORD_DESTINATIONS:
        value_norm = normalize_text(TransferDestination[name].value)
        if re.search(rf"\b{re.escape(value_norm)}\b", norm):
            return name, None
    if re.search(r"\bcatalogue\b", norm):
        # Bare "catalogue" mention with no app named - defaults to the ticket's own application's
        # config team (this org calls its paramétrage/configuration team "catalogue").
        return TEAM_QUALIFIED_APPS[default_catalogue_app][0], None
    # Bare app mention with no support/paramétrage qualifier maps to that app's support team.
    for app, (config_name, support_name) in TEAM_QUALIFIED_APPS.items():
        app_norm = normalize_text(app)
        if re.search(rf"\b{app_norm}\b", norm):
            return support_name, None
    return None, None


def match_transfer_destination(
    text: str | float, default_catalogue_app: str
) -> tuple[str | None, str | None]:
    """Matches actions_realisees free text to a TransferDestination enum member name.

    `default_catalogue_app` is the application of the ticket being processed ("FCI" or
    "COLORIS") - used to resolve a bare "catalogue" mention (no app named) to that app's config
    team.

    Returns (destination_name, flag_reason). flag_reason is set when the text mentions no
    destination at all.

    Some actions_realisees cells discuss more than one team/app before stating the actual
    destination (e.g. "...impact sur le paramétrage dans PROXIMA... Il n'y a pas d'incohérence
    dans le paramétrage COLORIS... Ticket transféré à Proxima."). To avoid matching an
    incidental earlier mention instead of the actual stated destination, destinations are first
    searched in a window right after each "transfér*" occurrence, most recent first (the last
    action log entry is normally the one that states the outcome); only if no such window yields
    a match does the search fall back to the whole text.
    """
    if not isinstance(text, str) or not text.strip():
        return None, "no_actions_text"
    norm = normalize_text(text)

    anchors = [m.start() for m in re.finditer(r"transf\w*", norm)]
    for start in reversed(anchors):
        dest, flag = _detect_destination(norm[start : start + 100], default_catalogue_app)
        if dest is not None or flag is not None:
            return dest, flag

    # "Banco" is also a common root-cause mention in resolved tickets, so it only counts as a
    # destination here when it appears right after a "transfer(red)" word.
    if re.search(r"transfer\w*.{0,15}banco", norm):
        return "BANCO", None

    dest, flag = _detect_destination(norm, default_catalogue_app)
    if dest is not None or flag is not None:
        return dest, flag

    # No destination stated in the text at all.
    return "DEVELOPMENT_TEAM", None


def parse_offer_version(raw: str | float) -> tuple[str | None, str | None]:
    """Parses the 'Offre composée' column into (offer, version).

    Multiple slash-separated offers keep only the first. A trailing "V<n>" token (possibly
    followed by more text, e.g. "V14 et V16") is taken as the version; everything before it is
    the offer. Offer names that don't match an Offer enum member, and version values that don't
    match a Version enum member, are still returned as-is - the enums are incomplete and are
    expected to be extended separately.
    """
    if not isinstance(raw, str) or not raw.strip():
        return None, None
    text = raw.strip()
    if "/" in text:
        text = text.split("/")[0].strip()
    match = re.search(r"\bV(\d+)\b", text, flags=re.IGNORECASE)
    if match:
        offer = text[: match.start()].strip()
        return (offer or None), f"V{match.group(1)}"
    return text, None

## FCI preprocessing

Raw export is semicolon-separated, UTF-8 with a BOM (`sep=";"`, `encoding="utf-8-sig"`).

`Date de livraison JIRA` and `Unnamed: 13` are 100% null (317/317) and are dropped as export artifacts.
Blank export rows (37 of them) are identified by a null `Statut` — every real ticket row has one,
so this is a more reliable filter than requiring both ID columns to be present (some real rows are
missing one of the two IDs but still have a status).

In [ ]:
fci = fci.drop(columns=["Date de livraison JIRA", "Unnamed: 13"])
fci = fci.dropna(subset=["Statut"]).reset_index(drop=True)

fci = fci.rename(columns={
    "ID Ticket GENERGY ": "genergy_id",
    "ID Ticket OCEANE": "oceane_id",
    "Description": "description",
    "Acteur": "acteur",
    "Paramétrage": "team_raw",
    "Statut": "statut_raw",
    "Priorité": "priorite_raw",
    "Actions réalisés": "actions_realisees",
    "Date d'ouverture": "date_ouverture_raw",
    "Date de résolution": "date_resolution_raw",
    "Durée de résolution": "duree_resolution_raw",
    "ID Jira": "jira_id",
})

print(fci.shape)
fci.head()

In [ ]:
# jira_id: strip stray trailing whitespace seen in raw values (e.g. "FCI-10066 ")
fci["jira_id"] = fci["jira_id"].str.strip()
fci["requires_jira"] = fci["jira_id"].notna()

fci[["jira_id", "requires_jira"]].value_counts(dropna=False)

In [ ]:
# Dates are dd/mm/yyyy, with a "dd/mm/yyyy HH:MM" variant on ~40 rows (all exactly 00:00 in this
# file, i.e. no real time-of-day info) - format="mixed" handles both.
fci["date_ouverture"] = pd.to_datetime(fci["date_ouverture_raw"], format="mixed", dayfirst=True)
fci["date_resolution"] = pd.to_datetime(fci["date_resolution_raw"], format="mixed", dayfirst=True)

fci[["date_ouverture_raw", "date_ouverture", "date_resolution_raw", "date_resolution"]].head()

In [ ]:
# duree_resolution_raw is a derived/display column, not a Ticket field - parsed only to
# cross-check consistency against (date_resolution - date_ouverture).
parsed = fci["duree_resolution_raw"].apply(parse_duree_resolution)
fci["duree_action"] = parsed.apply(lambda t: t[0])
fci["duree_jours"] = parsed.apply(lambda t: t[1])

# Cross-check 1: action word in the duration string should agree with Statut
action_to_statut = {"Résolu": "Résolu", "Transféré": "Transféré"}
mismatch_statut = fci[fci["duree_action"].map(action_to_statut) != fci["statut_raw"]]
print("Statut / durée action mismatches:", len(mismatch_statut))

# Cross-check 2: parsed day count vs actual calendar day difference
fci["date_diff_jours"] = (fci["date_resolution"] - fci["date_ouverture"]).dt.days
mismatch_days = fci[fci["duree_jours"] != fci["date_diff_jours"]]
print("Duration / date-diff mismatches:", len(mismatch_days))
mismatch_days[["genergy_id", "date_ouverture", "date_resolution", "duree_resolution_raw", "date_diff_jours"]]

In [ ]:
# Priorité values (P1-P4) already match Priority enum values exactly - direct passthrough.
assert set(fci["priorite_raw"].unique()) <= {"P1", "P2", "P3", "P4"}
fci["priority"] = fci["priorite_raw"]

# Statut -> Status enum. Raw data only ever has these two terminal states.
STATUT_MAP = {"Résolu": "RESOLVED", "Transféré": "TRANSFERRED"}
assert set(fci["statut_raw"].unique()) <= set(STATUT_MAP)
fci["status"] = fci["statut_raw"].map(STATUT_MAP)

# "Paramétrage" column actually encodes functional_team, not a free category:
# its only two values are "Support FCI" (-> SUPPORT) and "Paramétrage" (-> CONFIGURATION),
# matching the TransferDestination naming (SUPPORT_FCI / CONFIG_FCI).
TEAM_MAP = {"Support FCI": "SUPPORT", "Paramétrage": "CONFIGURATION"}
assert set(fci["team_raw"].unique()) <= set(TEAM_MAP)
fci["functional_team"] = fci["team_raw"].map(TEAM_MAP)

fci[["priorite_raw", "priority", "statut_raw", "status", "team_raw", "functional_team"]].drop_duplicates()

### `duree_resolution_raw`

Derived/display column (emoji + action word + day count), e.g. "✅ Résolu en 23 jours" /
"➡️ Transféré en < 1 jour". Not a `Ticket` field — parsed only to cross-check against
`date_resolution - date_ouverture`. The two disagree on 81/280 rows; the parsed day count is
consistently lower on cross-week-boundary spans, indicating `duree_resolution_raw` counts business
days while the date columns are calendar days. Not stored downstream.

### Field mapping decisions

- `title` — no separate title field in the source; built by truncating `description`
  (`TITLE_MAX_LEN`), full text kept in `description`.
- `category` — no source column for it in the FCI export; fixed to `Category.CATEGORY_VIDE` for
  every row.
- `assignee_id` — `acteur` (raw person name) is kept as-is; mapping names to Auth `User` UUIDs is
  the seeding script's responsibility, not this notebook's, to keep this output free of any
  infra/DB dependency.
- `transferred_to` — parsed from `actions_realisees` free text (see below); no destination column
  exists in the source.
- `resolution_notes` — `actions_realisees` copied as one string; not split into `Comment` entities.
- Final `status` — every row is migrated as `CLOSED` regardless of original Résolu/Transféré;
  `original_status` retains the source distinction.
- `created_at`/`resolved_at`/`closed_at` — localized to `Europe/Paris`; the source date columns
  have no real time-of-day in this file (all midnight where a time is present at all).

In [ ]:
fci["title"] = fci["description"].apply(truncate_title)

# category has no source column in the FCI export - fixed default for every row.
fci["category"] = "vide"

fci[["description", "title", "category"]].head()

### `transferred_to` matching rules

Destinations found in `actions_realisees` text across the 63 `Transféré` rows:
- "paramétrage/parametrage COLORIS" → `CONFIG_COLORIS`; "paramétrage/parametrage FCI" → `CONFIG_FCI`;
  "support COLORIS" → `SUPPORT_COLORIS`; "EEP" → `EEP`; "ACACIA" → `ACACIA`; "Habilitation" →
  `HABILITATION`; "transféré ... à BANCO" → `BANCO`.
- Bare "COLORIS" mentions with no support/paramétrage qualifier → `SUPPORT_COLORIS`.
- "Catalogue" is this org's own name for the paramétrage/configuration team: "catalogue COLORIS" →
  `CONFIG_COLORIS`, "catalogue FCI" → `CONFIG_FCI` (either word order), matched the same way as
  "paramétrage <app>". A bare "catalogue" mention with no app named defaults to this file's own
  application (`CONFIG_FCI` here).
- Rows where the text never states a destination (e.g. INC001010947598, discusses a Banco-related
  root cause with no "transferred to X" wording) are routed to `DEVELOPMENT_TEAM`.

In [ ]:
transferred_mask = fci["status"] == "TRANSFERRED"
results = fci.loc[transferred_mask, "actions_realisees"].apply(
    lambda text: match_transfer_destination(text, default_catalogue_app="FCI")
)
fci["transferred_to"] = None
fci["transfer_destination_flag"] = None
fci.loc[transferred_mask, "transferred_to"] = results.apply(lambda t: t[0])
fci.loc[transferred_mask, "transfer_destination_flag"] = results.apply(lambda t: t[1])

print(fci.loc[transferred_mask, "transferred_to"].value_counts(dropna=False))
print()
print(fci.loc[transferred_mask, "transfer_destination_flag"].value_counts(dropna=False))

### Final status and timestamps

Every migrated row is closed regardless of the original Résolu/Transféré distinction:
`status = CLOSED`, `closed_at = date_resolution`. `original_status` retains RESOLVED/TRANSFERRED,
and `resolved_at`/`transferred_to` are populated only for rows that actually went through that path
(matching what `Ticket.resolve()`/`Ticket.transfer()` would set before `close()`).

`resolution_notes` is `actions_realisees` copied as-is (applies to both resolved and transferred
rows — it's a general action log, not resolution-specific text).

In [ ]:
fci["original_status"] = fci["status"]  # RESOLVED / TRANSFERRED, kept for traceability
fci["status"] = "CLOSED"

fci["created_at"] = fci["date_ouverture"].dt.tz_localize("Europe/Paris")
fci["closed_at"] = fci["date_resolution"].dt.tz_localize("Europe/Paris")
fci["resolved_at"] = fci["closed_at"].where(fci["original_status"] == "RESOLVED")
fci["updated_at"] = fci["closed_at"]

fci["resolution_notes"] = fci["actions_realisees"]

fci[[
    "genergy_id", "original_status", "status", "created_at", "resolved_at",
    "closed_at", "transferred_to",
]].head(10)

### Enum value consistency

All enum-shaped columns are written using the domain enum's *value* string (e.g. `priority = "P4"`
= `Priority.LOW.value`, `category = "vide"` = `Category.CATEGORY_VIDE.value`, `status = "CLOSED"` =
`Status.CLOSED.value`). `transferred_to` was written as the enum *member name* by the matcher above,
so it's mapped to its value here (`DEST_NAME_TO_VALUE`, defined in Shared helpers) for the same
convention.

In [ ]:
fci["transferred_to"] = fci["transferred_to"].map(DEST_NAME_TO_VALUE)

fci.loc[fci["original_status"] == "TRANSFERRED", ["genergy_id", "transferred_to", "transfer_destination_flag"]]

## Final assembly and export

Assembles the processed columns into a shape close to the `Ticket` entity.

Fields that aren't a 1:1 mapping to `Ticket`:
- `acteur` is the raw name string, not `assignee_id` — UUID mapping happens in the seeding script.
- `original_status` and `transfer_destination_flag` are diagnostic columns, not `Ticket` fields —
  they preserve the source Résolu/Transféré distinction and mark rows the seeding script needs to
  handle specially (none remain flagged for FCI).
- `application` is fixed to `"FCI"`; `offer`/`version`/`element`/`vio_app` are `None` (not
  applicable outside COLORIS/AERO/VIO); `operational_highlight` defaults to `False` (no source
  signal); `archived_at` is `None` (nothing in this export represents archival).

In [ ]:
fci_processed = pd.DataFrame({
    "genergy_id": fci["genergy_id"],
    "oceane_id": fci["oceane_id"],
    "title": fci["title"],
    "description": fci["description"],
    "application": "FCI",
    "status": fci["status"],
    "priority": fci["priority"],
    "category": fci["category"],
    "functional_team": fci["functional_team"],
    "acteur": fci["acteur"],
    "created_at": fci["created_at"],
    "updated_at": fci["updated_at"],
    "resolved_at": fci["resolved_at"],
    "closed_at": fci["closed_at"],
    "resolution_notes": fci["resolution_notes"],
    "transferred_to": fci["transferred_to"],
    "jira_id": fci["jira_id"],
    "requires_jira": fci["requires_jira"],
    "jira_delivery_date": None,
    "operational_highlight": False,
    "offer": None,
    "version": None,
    "element": None,
    "vio_app": None,
    "archived_at": None,
    # diagnostics for the seeding script / manual review - not Ticket fields
    "original_status": fci["original_status"],
    "transfer_destination_flag": fci["transfer_destination_flag"],
})

print(fci_processed.shape)
fci_processed.info()

In [ ]:
DATA_PROCESSED_DIR = "../../data/processed/"

fci_processed.to_json(
    DATA_PROCESSED_DIR + "FCI.json", orient="records", date_format="iso", indent=2, force_ascii=False
)
print(f"Wrote {len(fci_processed)} rows to {DATA_PROCESSED_DIR}FCI.json")

`to_json(date_format="iso")` serializes tz-aware timestamps as UTC instants with a `Z` suffix, so
midnight `Europe/Paris` on 2025-10-03 (CEST, UTC+2) shows up as `"2025-10-02T22:00:00.000Z"` — same
instant, different displayed date.

### Remaining flagged rows

None: every `Transféré` row's `actions_realisees` states a destination (including the "catalogue"
mentions, now resolved to `CONFIG_FCI` per the matching rules above), so `transfer_destination_flag`
is `null` for all FCI rows.

In [ ]:
fci_processed[fci_processed["transfer_destination_flag"].notna()][
    ["genergy_id", "transfer_destination_flag", "resolution_notes"]
]

## COLORIS preprocessing

Same shape as FCI with two differences: the team column is named `Colonne1` instead of
`Paramétrage`, and there are two extra columns, `Offre composée` and `Catégorie d'incident`.

Blank export rows (23 of them) are dropped via null `Statut`, same as FCI.

In [ ]:
coloris = coloris.dropna(subset=["Statut"]).reset_index(drop=True)

coloris = coloris.rename(columns={
    "ID Ticket GENERGY ": "genergy_id",
    "ID Ticket OCEANE": "oceane_id",
    "Description": "description",
    "Acteur": "acteur",
    "Colonne1": "team_raw",
    "Statut": "statut_raw",
    "Priorité": "priorite_raw",
    "Actions réalisés": "actions_realisees",
    "Date d'ouverture": "date_ouverture_raw",
    "Date de résolution": "date_resolution_raw",
    "Durée de résolution": "duree_resolution_raw",
    "ID Jira": "jira_id",
    "Offre composée": "offre_composee_raw",
    "Catégorie d'incident": "categorie_incident_raw",
})

print(coloris.shape)
coloris.head()

In [ ]:
coloris["jira_id"] = coloris["jira_id"].str.strip()
coloris["requires_jira"] = coloris["jira_id"].notna()

coloris[["jira_id", "requires_jira"]].value_counts(dropna=False)

In [ ]:
# Unlike FCI, a handful of COLORIS rows carry a genuine intraday time (e.g. "07/04/2026 16:26")
# rather than a plain date - format="mixed" preserves it instead of discarding it.
coloris["date_ouverture"] = pd.to_datetime(coloris["date_ouverture_raw"], format="mixed", dayfirst=True)
coloris["date_resolution"] = pd.to_datetime(coloris["date_resolution_raw"], format="mixed", dayfirst=True)

coloris[["date_ouverture_raw", "date_ouverture", "date_resolution_raw", "date_resolution"]].head()

In [ ]:
parsed = coloris["duree_resolution_raw"].apply(parse_duree_resolution)
coloris["duree_action"] = parsed.apply(lambda t: t[0])
coloris["duree_jours"] = parsed.apply(lambda t: t[1])

action_to_statut = {"Résolu": "Résolu", "Transféré": "Transféré"}
mismatch_statut = coloris[coloris["duree_action"].map(action_to_statut) != coloris["statut_raw"]]
print("Statut / durée action mismatches:", len(mismatch_statut))

coloris["date_diff_jours"] = (coloris["date_resolution"] - coloris["date_ouverture"]).dt.days
mismatch_days = coloris[coloris["duree_jours"] != coloris["date_diff_jours"]]
print("Duration / date-diff mismatches:", len(mismatch_days))

In [ ]:
assert set(coloris["priorite_raw"].unique()) <= {"P1", "P2", "P3", "P4"}
coloris["priority"] = coloris["priorite_raw"]

STATUT_MAP = {"Résolu": "RESOLVED", "Transféré": "TRANSFERRED"}
assert set(coloris["statut_raw"].unique()) <= set(STATUT_MAP)
coloris["status"] = coloris["statut_raw"].map(STATUT_MAP)

TEAM_MAP = {"Support COLORIS": "SUPPORT", "Paramétrage COLORIS": "CONFIGURATION"}
assert set(coloris["team_raw"].unique()) <= set(TEAM_MAP)
coloris["functional_team"] = coloris["team_raw"].map(TEAM_MAP)

coloris[["priorite_raw", "priority", "statut_raw", "status", "team_raw", "functional_team"]].drop_duplicates()

### `title`, `category`, `offer`/`version`

`title` is built the same way as FCI (`truncate_title(description)`). One row
(INC001011250101) has a null `description` with no usable substitute and is dropped rather than
given a fabricated description.

`categorie_incident_raw` (113/309 non-null) is unreliable for most values: 76 rows just repeat the
team name ("Paramétrage COLORIS") and 17 repeat `Statut` ("Transféré"). Only "Bon usage" (20 rows)
is an actual `Category` value, so it's the only value mapped; everything else (including null)
defaults to `CATEGORY_VIDE`.

COLORIS tickets require both `offer` and `version` to be set, but `offre_composee_raw` is sparse
(107/309 rows) and there's no version column. `parse_offer_version` (Shared helpers) extracts both
from that single column: a trailing "V<n>" token becomes `version`, everything before it becomes
`offer`; multiple slash-separated offers keep the first; rows with no `offre_composee_raw` value
get `offer = version = None`. Parsed offer/version values aren't guaranteed to match the current
`Offer`/`Version` enum members — those enums are known to be incomplete and are expected to be
extended separately.

In [ ]:
coloris = coloris.dropna(subset=["description"]).reset_index(drop=True)

coloris["title"] = coloris["description"].apply(truncate_title)

coloris["category"] = coloris["categorie_incident_raw"].apply(
    lambda v: "Bon usage" if v == "Bon usage" else "vide"
)

offer_version = coloris["offre_composee_raw"].apply(parse_offer_version)
coloris["offer"] = offer_version.apply(lambda t: t[0])
coloris["version"] = offer_version.apply(lambda t: t[1])

coloris[["description", "title", "categorie_incident_raw", "category", "offre_composee_raw", "offer", "version"]].drop_duplicates(subset=["offre_composee_raw"])

In [ ]:
transferred_mask = coloris["status"] == "TRANSFERRED"
results = coloris.loc[transferred_mask, "actions_realisees"].apply(
    lambda text: match_transfer_destination(text, default_catalogue_app="COLORIS")
)
coloris["transferred_to"] = None
coloris["transfer_destination_flag"] = None
coloris.loc[transferred_mask, "transferred_to"] = results.apply(lambda t: t[0])
coloris.loc[transferred_mask, "transfer_destination_flag"] = results.apply(lambda t: t[1])

print(coloris.loc[transferred_mask, "transferred_to"].value_counts(dropna=False))
print()
print(coloris.loc[transferred_mask, "transfer_destination_flag"].value_counts(dropna=False))

In [ ]:
# INC001011254596 (acteur SOUISSI Amel) has no destination text in actions_realisees at all, so
# it's unmatched by match_transfer_destination ("no_actions_text"). Manually corrected: a second
# row for the same genergy_id (acteur GHARBI Anis, opened a day later) has
# team = "Support COLORIS" - so this transfer's real destination is SUPPORT_COLORIS. The genergy_id
# is duplicated across the two rows, so the mask is narrowed to the flagged (transferred, no-text)
# row specifically rather than every row sharing that id.
manual_override_mask = (coloris["genergy_id"] == "INC001011254596") & (
    coloris["transfer_destination_flag"] == "no_actions_text"
)
assert manual_override_mask.sum() == 1
coloris.loc[manual_override_mask, "transferred_to"] = "SUPPORT_COLORIS"
coloris.loc[manual_override_mask, "transfer_destination_flag"] = None

coloris.loc[
    manual_override_mask, ["genergy_id", "acteur", "transferred_to", "transfer_destination_flag"]
]

In [ ]:
coloris["original_status"] = coloris["status"]
coloris["status"] = "CLOSED"

coloris["created_at"] = coloris["date_ouverture"].dt.tz_localize("Europe/Paris")
coloris["closed_at"] = coloris["date_resolution"].dt.tz_localize("Europe/Paris")
coloris["resolved_at"] = coloris["closed_at"].where(coloris["original_status"] == "RESOLVED")
coloris["updated_at"] = coloris["closed_at"]

coloris["resolution_notes"] = coloris["actions_realisees"]

coloris["transferred_to"] = coloris["transferred_to"].map(DEST_NAME_TO_VALUE)

coloris[[
    "genergy_id", "original_status", "status", "created_at", "resolved_at",
    "closed_at", "transferred_to",
]].head(10)

## Final assembly and export

Same structure as FCI. `oceane_id`/`jira_delivery_date`/`element`/`vio_app` are `None`
(not applicable to COLORIS); `operational_highlight` defaults to `False`; `archived_at` is `None`.
`offer`/`version` come from the parsed `offre_composee_raw` column instead of being fixed `None`
like FCI.

In [ ]:
coloris_processed = pd.DataFrame({
    "genergy_id": coloris["genergy_id"],
    "oceane_id": coloris["oceane_id"],
    "title": coloris["title"],
    "description": coloris["description"],
    "application": "COLORIS",
    "status": coloris["status"],
    "priority": coloris["priority"],
    "category": coloris["category"],
    "functional_team": coloris["functional_team"],
    "acteur": coloris["acteur"],
    "created_at": coloris["created_at"],
    "updated_at": coloris["updated_at"],
    "resolved_at": coloris["resolved_at"],
    "closed_at": coloris["closed_at"],
    "resolution_notes": coloris["resolution_notes"],
    "transferred_to": coloris["transferred_to"],
    "jira_id": coloris["jira_id"],
    "requires_jira": coloris["requires_jira"],
    "jira_delivery_date": None,
    "operational_highlight": False,
    "offer": coloris["offer"],
    "version": coloris["version"],
    "element": None,
    "vio_app": None,
    "archived_at": None,
    "original_status": coloris["original_status"],
    "transfer_destination_flag": coloris["transfer_destination_flag"],
})

print(coloris_processed.shape)
coloris_processed.info()

In [ ]:
coloris_processed.to_json(
    DATA_PROCESSED_DIR + "COLORIS.json", orient="records", date_format="iso", indent=2, force_ascii=False
)
print(f"Wrote {len(coloris_processed)} rows to {DATA_PROCESSED_DIR}COLORIS.json")

## AERO preprocessing

Different shape from FCI/COLORIS: no `ID Ticket OCEANE` column, no team/perimeter column, and two
extra columns instead - `ELEMENT` (maps to the `Element` enum, required for AERO tickets) and
`Catégorie` (maps to `Category`, unlike FCI/COLORIS which have no reliable category signal).

`TYPE` is dropped: it's `"AERO"` for all but one row (`"AERO GP"`), redundant with the fixed
`application = "AERO"`.

Blank export rows (139 of them) are dropped via null `Statut`, same convention as FCI/COLORIS.

Every row has `Statut = "Résolu"` - AERO has no `Transféré` rows in this export, so there's no
`transferred_to` matching step for this application.

`ID Ticket GENERGY` is literally `"mail"` on 6 rows (reported by email rather than through the
ticket system) - kept as-is rather than nulled out, per prior decision.

In [ ]:
aero = aero.dropna(subset=["Statut"]).reset_index(drop=True)

aero = aero.rename(columns={
    "ID Ticket GENERGY ": "genergy_id",
    "Description": "description",
    "Acteur": "acteur",
    "Catégorie ": "categorie_raw",
    "Statut": "statut_raw",
    "Priorité": "priorite_raw",
    "Actions réalisés": "actions_realisees",
    "Date d'ouverture": "date_ouverture_raw",
    "Date de résolution": "date_resolution_raw",
    "Durée de résolution": "duree_resolution_jours",
    "ID Jira": "jira_id",
    "ELEMENT": "element_raw",
})
aero = aero.drop(columns=["TYPE"])

print(aero.shape)
aero.head()

In [ ]:
aero["jira_id"] = aero["jira_id"].str.strip()
aero["requires_jira"] = aero["jira_id"].notna()

aero[["jira_id", "requires_jira"]].value_counts(dropna=False)

In [ ]:
aero["date_ouverture"] = pd.to_datetime(aero["date_ouverture_raw"], format="mixed", dayfirst=True)
aero["date_resolution"] = pd.to_datetime(aero["date_resolution_raw"], format="mixed", dayfirst=True)

# duree_resolution_jours is already a plain day count here (not the emoji-prefixed FCI/COLORIS
# string) - cross-check directly against the calendar day difference.
aero["date_diff_jours"] = (aero["date_resolution"] - aero["date_ouverture"]).dt.days
mismatch_days = aero[aero["duree_resolution_jours"] != aero["date_diff_jours"]]
print("Duration / date-diff mismatches:", len(mismatch_days))

aero[["date_ouverture_raw", "date_ouverture", "date_resolution_raw", "date_resolution", "duree_resolution_jours", "date_diff_jours"]].head()

In [ ]:
assert set(aero["priorite_raw"].unique()) <= {"P1", "P2", "P3", "P4"}
aero["priority"] = aero["priorite_raw"]

assert set(aero["statut_raw"].unique()) == {"Résolu"}
aero["status"] = "RESOLVED"

# No team/perimeter column in this export - fixed to SUPPORT for every AERO row.
aero["functional_team"] = "SUPPORT"

aero[["priorite_raw", "priority", "statut_raw", "status", "functional_team"]].drop_duplicates()

### `title`, `element`, `category`

`title` is built the same way as FCI/COLORIS.

`element_raw` matches the `Element` enum values after normalizing case/accents/dashes, except one
typo ("INATENDU" instead of "INATTENDU") corrected explicitly.

`categorie_raw` maps to `Category` with best-effort semantic matches plus two explicit overrides:
"Aucun problème lié à AERO" → `CATEGORY_HORS_PERIMETRE` (same as "Hors périmètre d'AERO") and
"Infrastructre" → `CATEGORY_INFRASTRUCTURE` (enum member added for this).

In [ ]:
from app.modules.ticket_management.domain.enums.element import Element


def _normalize_element(text: str) -> str:
    return normalize_text(text).replace("-", " ").strip()


ELEMENT_BY_NORM = {_normalize_element(e.value): e.name for e in Element}
# Source typo: "INATENDU" instead of "INATTENDU".
ELEMENT_BY_NORM[_normalize_element("MESSAGE D'ERREUR OU RESULTAT INATENDU")] = "ERREUR"

aero["element"] = aero["element_raw"].apply(
    lambda v: Element[ELEMENT_BY_NORM[_normalize_element(v)]].value
)

CATEGORIE_MAP = {
    "Bug": "Bug",
    "Demande d'assistance": "Assistance client",
    "Hors perimètre d'AERO": "Hors périmètre",
    "Information Manquante": "Manque d'information",
    "Aucun probléme lié a AERO ": "Hors périmètre",
    "Infrastructre": "Infrastructure",
}
assert set(aero["categorie_raw"].unique()) <= set(CATEGORIE_MAP)
aero["category"] = aero["categorie_raw"].map(CATEGORIE_MAP)

aero["title"] = aero["description"].apply(truncate_title)

aero[["element_raw", "element", "categorie_raw", "category"]].drop_duplicates()

In [ ]:
aero["original_status"] = aero["status"]
aero["status"] = "CLOSED"

aero["created_at"] = aero["date_ouverture"].dt.tz_localize("Europe/Paris")
aero["closed_at"] = aero["date_resolution"].dt.tz_localize("Europe/Paris")
aero["resolved_at"] = aero["closed_at"]
aero["updated_at"] = aero["closed_at"]

aero["resolution_notes"] = aero["actions_realisees"]
aero["transferred_to"] = None
aero["transfer_destination_flag"] = None

aero[["genergy_id", "original_status", "status", "created_at", "resolved_at", "closed_at"]].head(10)

## Final assembly and export

Same structure as the other applications. `oceane_id` is `None` (no such column in this export);
`offer`/`version`/`vio_app` are `None` (not applicable to AERO); `jira_delivery_date` is `None`;
`operational_highlight` defaults to `False`; `archived_at` is `None`.

In [ ]:
aero_processed = pd.DataFrame({
    "genergy_id": aero["genergy_id"],
    "oceane_id": None,
    "title": aero["title"],
    "description": aero["description"],
    "application": "AERO",
    "status": aero["status"],
    "priority": aero["priority"],
    "category": aero["category"],
    "functional_team": aero["functional_team"],
    "acteur": aero["acteur"],
    "created_at": aero["created_at"],
    "updated_at": aero["updated_at"],
    "resolved_at": aero["resolved_at"],
    "closed_at": aero["closed_at"],
    "resolution_notes": aero["resolution_notes"],
    "transferred_to": aero["transferred_to"],
    "jira_id": aero["jira_id"],
    "requires_jira": aero["requires_jira"],
    "jira_delivery_date": None,
    "operational_highlight": False,
    "offer": None,
    "version": None,
    "element": aero["element"],
    "vio_app": None,
    "archived_at": None,
    "original_status": aero["original_status"],
    "transfer_destination_flag": aero["transfer_destination_flag"],
})

print(aero_processed.shape)
aero_processed.info()

In [ ]:
aero_processed.to_json(
    DATA_PROCESSED_DIR + "AERO.json", orient="records", date_format="iso", indent=2, force_ascii=False
)
print(f"Wrote {len(aero_processed)} rows to {DATA_PROCESSED_DIR}AERO.json")

## VIO preprocessing

Simpler shape than FCI/COLORIS: `App` maps to the `VioApp` enum (required for VIO tickets),
`Périmètre` is constantly `"Support VIO"` (→ `functional_team = SUPPORT` for every row), `Statut`
is constantly `"Résolu"` (no transfers, so no `transferred_to` matching step), and there's no
category column at all (defaults to `CATEGORY_VIDE`, same as FCI).

Blank export rows (10 of them) are dropped via null `Statut`, same convention as the other files.

In [ ]:
vio = vio.dropna(subset=["Statut"]).reset_index(drop=True)

vio = vio.rename(columns={
    "ID Ticket GENERGY ": "genergy_id",
    "ID Ticket OCEANE": "oceane_id",
    "Description": "description",
    "Acteur": "acteur",
    "App": "vio_app_raw",
    "Périmètre ": "perimetre_raw",
    "Statut": "statut_raw",
    "Priorité": "priorite_raw",
    "Actions réalisés": "actions_realisees",
    "Date d'ouverture": "date_ouverture_raw",
    "Date de résolution": "date_resolution_raw",
    "Durée de résolution": "duree_resolution_raw",
    "ID Jira": "jira_id",
})

print(vio.shape)
vio.head()

In [ ]:
vio["jira_id"] = vio["jira_id"].str.strip()
vio["requires_jira"] = vio["jira_id"].notna()

vio[["jira_id", "requires_jira"]].value_counts(dropna=False)

One row (`ID Ticket OCEANE` `2512L01771`) has `Date de résolution = "19/01/1900"`, an obviously
bogus placeholder (opened 2025-12-12, duration text says "< 1 jour"). Per prior decision, its
resolution date is substituted with its opening date rather than kept as-is or left null.

In [ ]:
bogus_resolution_date = vio["date_resolution_raw"] == "19/01/1900"
vio.loc[bogus_resolution_date, "date_resolution_raw"] = vio.loc[bogus_resolution_date, "date_ouverture_raw"]

vio["date_ouverture"] = pd.to_datetime(vio["date_ouverture_raw"], format="mixed", dayfirst=True)
vio["date_resolution"] = pd.to_datetime(vio["date_resolution_raw"], format="mixed", dayfirst=True)

parsed = vio["duree_resolution_raw"].apply(parse_duree_resolution)
vio["duree_jours"] = parsed.apply(lambda t: t[1])
vio["date_diff_jours"] = (vio["date_resolution"] - vio["date_ouverture"]).dt.days
mismatch_days = vio[vio["duree_jours"] != vio["date_diff_jours"]]
print("Duration / date-diff mismatches:", len(mismatch_days))

vio[["date_ouverture_raw", "date_ouverture", "date_resolution_raw", "date_resolution"]].head()

In [ ]:
from app.modules.ticket_management.domain.enums.vio_app import VioApp

assert set(vio["priorite_raw"].unique()) <= {"P1", "P2", "P3", "P4"}
vio["priority"] = vio["priorite_raw"]

assert set(vio["statut_raw"].unique()) == {"Résolu"}
vio["status"] = "RESOLVED"

assert set(vio["perimetre_raw"].unique()) == {"Support VIO"}
vio["functional_team"] = "SUPPORT"

VIO_APP_MAP = {"Parc": "PARC", "Vigie": "VIGIE", "Sagic": "SAGIC", "Fop ": "FOP"}
assert set(vio["vio_app_raw"].unique()) <= set(VIO_APP_MAP)
vio["vio_app"] = vio["vio_app_raw"].map(VIO_APP_MAP).map(lambda name: VioApp[name].value)

vio["title"] = vio["description"].apply(truncate_title)
vio["category"] = "vide"

vio[["vio_app_raw", "vio_app", "priority", "status", "functional_team", "category"]].drop_duplicates()

In [ ]:
vio["original_status"] = vio["status"]
vio["status"] = "CLOSED"

vio["created_at"] = vio["date_ouverture"].dt.tz_localize("Europe/Paris")
vio["closed_at"] = vio["date_resolution"].dt.tz_localize("Europe/Paris")
vio["resolved_at"] = vio["closed_at"]
vio["updated_at"] = vio["closed_at"]

vio["resolution_notes"] = vio["actions_realisees"]
vio["transferred_to"] = None
vio["transfer_destination_flag"] = None

vio[["oceane_id", "original_status", "status", "created_at", "resolved_at", "closed_at", "vio_app"]].head(10)

## Final assembly and export

Same structure as the other applications. `offer`/`version`/`element` are `None` (not applicable
to VIO); `jira_delivery_date` is `None`; `operational_highlight` defaults to `False`;
`archived_at` is `None`.

In [ ]:
vio_processed = pd.DataFrame({
    "genergy_id": vio["genergy_id"],
    "oceane_id": vio["oceane_id"],
    "title": vio["title"],
    "description": vio["description"],
    "application": "VIO",
    "status": vio["status"],
    "priority": vio["priority"],
    "category": vio["category"],
    "functional_team": vio["functional_team"],
    "acteur": vio["acteur"],
    "created_at": vio["created_at"],
    "updated_at": vio["updated_at"],
    "resolved_at": vio["resolved_at"],
    "closed_at": vio["closed_at"],
    "resolution_notes": vio["resolution_notes"],
    "transferred_to": vio["transferred_to"],
    "jira_id": vio["jira_id"],
    "requires_jira": vio["requires_jira"],
    "jira_delivery_date": None,
    "operational_highlight": False,
    "offer": None,
    "version": None,
    "element": None,
    "vio_app": vio["vio_app"],
    "archived_at": None,
    "original_status": vio["original_status"],
    "transfer_destination_flag": vio["transfer_destination_flag"],
})

print(vio_processed.shape)
vio_processed.info()

In [ ]:
vio_processed.to_json(
    DATA_PROCESSED_DIR + "VIO.json", orient="records", date_format="iso", indent=2, force_ascii=False
)
print(f"Wrote {len(vio_processed)} rows to {DATA_PROCESSED_DIR}VIO.json")

## Conclusion and flagged notes:

- ``FCI``: 280 rows, 0 flagged — "catalogue" mentions resolved to `CONFIG_FCI`/`CONFIG_COLORIS`
  (catalogue = paramétrage)
- ``COLORIS``: 285 rows, 0 flagged — 1 catalogue mention resolved to `CONFIG_FCI`; 1 row with no
  destination text manually corrected to `SUPPORT_COLORIS` (per a duplicate row for the same
  ticket, created a day later, with team = "Support COLORIS") — offer/version parsed from the
  composite offer column
- ``AERO``: 54 rows, element/category mapped from dedicated columns, 6 "mail"-origin rows preserved as-is
- ``VIO``: 180 rows, vio_app mapped, one bogus 1900 date corrected